# Round 2 – Smart Irrigation Requirement Prediction

**Algorithm:** Decision Tree  
**Dataset:** `Dataset/train.csv`

This notebook contains the complete Round 2 workflow in separate cells:
- Feature Engineering
- Feature Selection & Preprocessing
- Train/Test Split
- Decision Tree Training
- Evaluation
- Error Analysis & Feature Importance
- Model Saving


In [1]:
import pandas as pd
import numpy as np

# ==========================================
# ROUND 2 - DATASET LOADING
# ==========================================

df = pd.read_csv("Dataset/train.csv")

print("=" * 60)
print("ROUND 2 - FEATURE ENGINEERING")
print("=" * 60)

print("\nOriginal Dataset Shape:")
print(df.shape)

print("\nTarget:")
print(df["Irrigation_Need"].value_counts())


ROUND 2 - FEATURE ENGINEERING

Original Dataset Shape:
(630000, 21)

Target:
Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64


In [2]:
# ==========================================
# FEATURE ENGINEERING
# ==========================================

print("\n" + "=" * 60)
print("CREATING ENGINEERED FEATURES")
print("=" * 60)

# 1. Soil Moisture Deficit
df["Moisture_Deficit"] = 100 - df["Soil_Moisture"]

# 2. Temperature-Humidity Interaction
df["Temperature_Humidity_Index"] = (
    df["Temperature_C"] * (100 - df["Humidity"])
)

# 3. Heat-Sunlight Stress
df["Heat_Sunlight_Stress"] = (
    df["Temperature_C"] * df["Sunlight_Hours"]
)

# 4. Rainfall and Previous Irrigation
df["Water_Availability"] = (
    df["Rainfall_mm"] + df["Previous_Irrigation_mm"]
)

# 5. Evaporation Risk
df["Evaporation_Risk"] = (
    df["Temperature_C"]
    + df["Wind_Speed_kmh"]
    + df["Sunlight_Hours"]
    - df["Humidity"] / 10
)

# 6. Moisture-Rainfall Stress
df["Moisture_Rainfall_Stress"] = (
    (100 - df["Soil_Moisture"])
    / (df["Rainfall_mm"] + 1)
)

# 7. Irrigation History Ratio
df["Previous_Irrigation_Ratio"] = (
    df["Previous_Irrigation_mm"]
    / (df["Rainfall_mm"] + 1)
)

engineered_features = [
    "Moisture_Deficit",
    "Temperature_Humidity_Index",
    "Heat_Sunlight_Stress",
    "Water_Availability",
    "Evaporation_Risk",
    "Moisture_Rainfall_Stress",
    "Previous_Irrigation_Ratio"
]

print("\nEngineered Features Created:")

for feature in engineered_features:
    print("-", feature)

print("\nNew Dataset Shape:")
print(df.shape)

print("\nEngineered Feature Sample:")
print(df[engineered_features].head())



CREATING ENGINEERED FEATURES

Engineered Features Created:
- Moisture_Deficit
- Temperature_Humidity_Index
- Heat_Sunlight_Stress
- Water_Availability
- Evaporation_Risk
- Moisture_Rainfall_Stress
- Previous_Irrigation_Ratio

New Dataset Shape:
(630000, 28)

Engineered Feature Sample:
   Moisture_Deficit  Temperature_Humidity_Index  Heat_Sunlight_Stress  \
0             67.42                    741.3439               88.5590   
1             43.39                    736.6488              159.9816   
2             72.29                    209.8266              163.1685   
3             86.68                    511.8876              121.4784   
4             40.86                    179.7558              140.5290   

   Water_Availability  Evaporation_Risk  Moisture_Rainfall_Stress  \
0              838.15            32.639                  0.092739   
1             1032.82            26.504                  0.043977   
2             2312.08            27.648                  0.032819  

In [3]:
# ==========================================
# FEATURE SELECTION & PREPROCESSING
# ==========================================

from sklearn.preprocessing import OneHotEncoder

print("\n" + "=" * 60)
print("FEATURE SELECTION & PREPROCESSING")
print("=" * 60)

target_column = "Irrigation_Need"

# Remove ID because it is only an identifier
features = df.drop(columns=[target_column, "id"])
y = df[target_column]

# Identify categorical and numerical features
categorical_features = features.select_dtypes(
    include=["object", "str"]
).columns.tolist()

numerical_features = features.select_dtypes(
    include=["number"]
).columns.tolist()

print("\nNumerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

print("\nNumber of Numerical Features:", len(numerical_features))
print("Number of Categorical Features:", len(categorical_features))

# One-hot encode categorical variables
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoded_categorical = encoder.fit_transform(
    features[categorical_features]
)

encoded_columns = encoder.get_feature_names_out(
    categorical_features
)

encoded_df = pd.DataFrame(
    encoded_categorical,
    columns=encoded_columns,
    index=features.index
)

# Combine numerical + encoded categorical features
X = pd.concat(
    [
        features[numerical_features].reset_index(drop=True),
        encoded_df.reset_index(drop=True)
    ],
    axis=1
)

print("\nFinal Feature Matrix Shape:")
print(X.shape)

print("\nTarget Shape:")
print(y.shape)

print("\nFinal Feature Sample:")
print(X.head())

print("\nTarget Sample:")
print(y.head())



FEATURE SELECTION & PREPROCESSING

Numerical Features:
['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm', 'Moisture_Deficit', 'Temperature_Humidity_Index', 'Heat_Sunlight_Stress', 'Water_Availability', 'Evaporation_Risk', 'Moisture_Rainfall_Stress', 'Previous_Irrigation_Ratio']

Categorical Features:
['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']

Number of Numerical Features: 18
Number of Categorical Features: 8

Final Feature Matrix Shape:
(630000, 50)

Target Shape:
(630000,)

Final Feature Sample:
   Soil_pH  Soil_Moisture  Organic_Carbon  Electrical_Conductivity  \
0     4.92          32.58            1.01                     3.05   
1     7.08          56.61            0.44                     2.00   
2     5.69          27.71            0.81                 

In [4]:
# ==========================================
# TRAIN / TEST SPLIT
# ==========================================

from sklearn.model_selection import train_test_split

print("\n" + "=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining Feature Shape:")
print(X_train.shape)

print("\nTesting Feature Shape:")
print(X_test.shape)

print("\nTraining Target Shape:")
print(y_train.shape)

print("\nTesting Target Shape:")
print(y_test.shape)

print("\nTraining Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())



TRAIN / TEST SPLIT

Training Feature Shape:
(504000, 50)

Testing Feature Shape:
(126000, 50)

Training Target Shape:
(504000,)

Testing Target Shape:
(126000,)

Training Target Distribution:
Irrigation_Need
Low       295934
Medium    191259
High       16807
Name: count, dtype: int64

Testing Target Distribution:
Irrigation_Need
Low       73983
Medium    47815
High       4202
Name: count, dtype: int64


In [5]:
# ==========================================
# DECISION TREE MODEL TRAINING
# ==========================================

from sklearn.tree import DecisionTreeClassifier

print("\n" + "=" * 60)
print("DECISION TREE MODEL TRAINING")
print("=" * 60)

model = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced",
    max_depth=12,
    min_samples_leaf=10
)

print("\nTraining Decision Tree...")

model.fit(X_train, y_train)

print("\nDecision Tree Training Completed!")

print("\nModel Parameters:")
print("Algorithm       : Decision Tree")
print("Max Depth       :", model.max_depth)
print("Min Samples Leaf:", model.min_samples_leaf)
print("Class Weight    :", model.class_weight)



DECISION TREE MODEL TRAINING

Training Decision Tree...

Decision Tree Training Completed!

Model Parameters:
Algorithm       : Decision Tree
Max Depth       : 12
Min Samples Leaf: 10
Class Weight    : balanced


In [6]:
# ==========================================
# MODEL PREDICTION & EVALUATION
# ==========================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("\n" + "=" * 60)
print("MODEL EVALUATION")
print("=" * 60)

# Predict on test data
y_pred = model.predict(X_test)

print("\nPrediction Completed!")

# Metrics
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

print("\n" + "-" * 60)
print("OVERALL PERFORMANCE")
print("-" * 60)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

print("\n" + "-" * 60)
print("CLASSIFICATION REPORT")
print("-" * 60)

print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

print("\n" + "-" * 60)
print("CONFUSION MATRIX")
print("-" * 60)

cm = confusion_matrix(y_test, y_pred)
print(cm)



MODEL EVALUATION

Prediction Completed!

------------------------------------------------------------
OVERALL PERFORMANCE
------------------------------------------------------------
Accuracy  : 0.9775
Precision : 0.9790
Recall    : 0.9775
F1 Score  : 0.9779

------------------------------------------------------------
CLASSIFICATION REPORT
------------------------------------------------------------
              precision    recall  f1-score   support

        High       0.77      0.94      0.85      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.98      0.96      0.97     47815

    accuracy                           0.98    126000
   macro avg       0.91      0.96      0.94    126000
weighted avg       0.98      0.98      0.98    126000


------------------------------------------------------------
CONFUSION MATRIX
------------------------------------------------------------
[[ 3969     0   233]
 [    4 73483   496]
 [ 1159   949 45707]]


In [7]:
# ==========================================
# ERROR ANALYSIS & FEATURE IMPORTANCE
# ==========================================

print("\n" + "=" * 60)
print("ERROR ANALYSIS & FEATURE IMPORTANCE")
print("=" * 60)

# Error analysis
errors = y_test != y_pred

print("\nTotal Test Samples:", len(y_test))
print("Correct Predictions:", (~errors).sum())
print("Incorrect Predictions:", errors.sum())

error_rate = errors.mean()

print(f"Error Rate: {error_rate:.4f}")

# Feature importance
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop 15 Important Features:")
print(
    feature_importance.head(15).to_string(index=False)
)

print("\n" + "-" * 60)
print("TOP 10 IMPORTANT FACTORS")
print("-" * 60)

for i, row in feature_importance.head(10).iterrows():
    print(
        f"{row['Feature']:<40} "
        f"{row['Importance']:.6f}"
    )



ERROR ANALYSIS & FEATURE IMPORTANCE

Total Test Samples: 126000
Correct Predictions: 123159
Incorrect Predictions: 2841
Error Rate: 0.0225

Top 15 Important Features:
                     Feature  Importance
               Soil_Moisture    0.284081
               Temperature_C    0.166569
                 Rainfall_mm    0.122544
              Wind_Speed_kmh    0.102429
Crop_Growth_Stage_Vegetative    0.088806
           Mulching_Used_Yes    0.052643
 Crop_Growth_Stage_Flowering    0.051913
            Mulching_Used_No    0.036857
    Crop_Growth_Stage_Sowing    0.028374
   Crop_Growth_Stage_Harvest    0.026815
            Evaporation_Risk    0.017784
          Water_Availability    0.008562
    Moisture_Rainfall_Stress    0.003426
          Field_Area_hectare    0.001147
                     Soil_pH    0.001119

------------------------------------------------------------
TOP 10 IMPORTANT FACTORS
------------------------------------------------------------
Soil_Moisture               

In [8]:
# ==========================================
# SAVE MODEL
# ==========================================

import joblib
import os

print("\n" + "=" * 60)
print("SAVING MODEL")
print("=" * 60)

# Create Models folder
os.makedirs("Models", exist_ok=True)

# Save Decision Tree
model_path = "Models/irrigation_decision_tree.pkl"

joblib.dump(model, model_path)

print("\nModel saved successfully!")
print("Saved Path:", model_path)

# Verify file
if os.path.exists(model_path):
    print("Model File Status: READY")
else:
    print("Model File Status: NOT FOUND")



SAVING MODEL

Model saved successfully!
Saved Path: Models/irrigation_decision_tree.pkl
Model File Status: READY
